In [1]:
import pandas as pd, json, re

In [11]:
from pathlib import Path

# Input Excel/CSV and output JSONL
IN_PATH  = Path("Book1.xlsx")            
OUT_PATH = Path("train_messages.jsonl")   

# Fixed column names
PROMPT_COL   = "Prompt"
RESPONSE_COL = "Response"

# test
ADD_SYSTEM = None 

In [12]:
def read_table(path: Path) -> pd.DataFrame:
    suf = path.suffix.lower()
    if suf in [".csv", ".tsv"]:
        sep = "," if suf == ".csv" else "\t"
        return pd.read_csv(path, sep=sep)
    if suf in [".xlsx", ".xls", ".xlsm"]:
        try:
            return pd.read_excel(path)
        except Exception as e:
            raise RuntimeError(
                f"Failed to read {path}. If Excel engines aren't available, "
                "export the sheet to CSV and set IN_PATH to that CSV."
            ) from e
    raise ValueError(f"Unsupported file type: {suf}")

In [13]:
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s = s.strip()
    s = re.sub(r'(?:<\|image\|>\s*)+', '', s)
    s = re.sub(r'(?:>。)+', '。', s)
    s = re.sub(r'\b(?:Human:|User:|Assistant:)\b', '', s, flags=re.I)
    s = re.sub(r'[ \t]+\n', '\n', s)
    s = re.sub(r'\n{3,}', '\n\n', s)
    return s.strip()

def to_qwen_messages_jsonl(df: pd.DataFrame, out_path: Path, add_system: str | None = None):
    for col in (PROMPT_COL, RESPONSE_COL):
        if col not in df.columns:
            raise KeyError(f"Missing required column: {col}. Columns present: {list(df.columns)}")

    n_total  = len(df)
    n_skip   = 0
    written  = 0

    with out_path.open("w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            user = clean_text(row[PROMPT_COL])
            assistant = clean_text(row[RESPONSE_COL])
            if not user or not assistant:
                n_skip += 1
                continue
            messages = []
            if add_system:
                messages.append({"role": "system", "content": add_system})
            messages.append({"role": "user", "content": user})
            messages.append({"role": "assistant", "content": assistant})
            f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
            written += 1

    return {"total": n_total, "skipped": n_skip, "written": written, "out": str(out_path)}


In [14]:
df = read_table(IN_PATH)

In [15]:
df

,Prompt,Response
0,请问一下成都适合家庭出游的旅游景点有哪些呢？,1. 成都大熊猫繁育研究基地：这是专门致力于大熊猫保护和研究的机构，园内大熊猫数量众多，孩子...
1,请问如果我想三日内游遍成都所有著名景点路线该如何规划？,第一天，上午前往四川博物院，这是西南最大博物馆，可重点参观巴蜀青铜馆、汉代陶石馆等，欣赏说唱...
2,请问一日游玩都江堰景区的开销一般是多少呢,一日游玩都江堰景区的开销一般在150元至350元之间，具体如下：从成都前往都江堰，若选择动车...
3,请问游遍成都著名景点的比较经济的路线有哪些,第一天：市区文化探索\n 上午前往武侯祠，这里是中国唯一君臣合祀祠庙，红墙竹影拍照很美，门票...
4,请问想要在成都充分领会四川特色文化应该去哪些景点？,想要在成都充分领悟四川特色文化，可以去以下景点：\n 武侯祠 ：全国重点文物保护单位，是中国...
...,...,...
112,想吃本地人常去的小吃，预算友好，并且离市区不远。,去“建设路小吃”区域：以实惠小店与地道口味著称。建议先小份多样化打卡（如钵钵鸡、凉粉、钟水饺...
113,带外地朋友想体验“烟火气”，避开景区价，有没有靠谱路线？,“建设路小吃”性价比高：选评分稳定的老店开局，接着去人气小店尝一两样主打；移动支付与排队号提...
114,夏天也想滑雪，是否有室内雪场？适合新手吗？,可以去“融创滑雪场”（室内雪世界）：全年恒温造雪，设初级/中级道与教学区，适合新手入门；支持...
115,想体验一次本地偶像女团的现场，最好互动感强、票价别太高，怎么安排更稳妥？,建议关注“CGT48”剧场/专场与小舞台活动：①票务以官方小程序/票务平台为准，优先选择视线...


In [16]:
stats = to_qwen_messages_jsonl(df, OUT_PATH, add_system=ADD_SYSTEM)
print(stats)

{'total': 117, 'skipped': 0, 'written': 117, 'out': 'train_messages.jsonl'}


In [17]:
from itertools import islice
with OUT_PATH.open("r", encoding="utf-8") as f:
    for line in islice(f, 3):
        print(line.strip())

{"messages": [{"role": "user", "content": "请问一下成都适合家庭出游的旅游景点有哪些呢？"}, {"role": "assistant", "content": "1. 成都大熊猫繁育研究基地：这是专门致力于大熊猫保护和研究的机构，园内大熊猫数量众多，孩子们可以近距离观察到大熊猫的憨态可掬，了解它们的生活习性，同时基地环境优美，适合全家一起散步游玩。 2. 都江堰景区 ：作为中华第一古堰，都江堰水利工程有着悠久的历史和伟大的智慧。家长可以带孩子参观宝瓶口引水口、飞沙堰泄洪坝、鱼嘴分水堤等，了解四六分水、二八淘沙的治水原理，寓教于乐。 3. 武侯祠 ：这里是三国迷的圣地，能让游客深入了解三国文化，园内的古建、历史遗迹、雕塑和石碑等元素丰富，观赏性和可玩性高，还可以在红墙竹影处拍摄武侠风亲子照。 4. 宽窄巷子：是成都的历史文化街8区，保留了众多老成都的生活风貌，街道相对平坦，适合带着老人和小孩漫步，大家可以在这里欣赏古建筑，品尝兔头、冒菜等地道的成都小吃。 5. 锦里古街：充满川西传统风情，街道两旁是仿古建筑，有各种特色小店和琳琅满目的小吃，能让全家人都愉快满足，还可以观看皮影戏、吹糖人等传统技艺表演。  6. 青城山：被誉为“青城天下幽”，是道教发源地之一，山清水秀，步道平缓，适合全家一起登山游玩，享受大自然的美好，呼吸新鲜空气。"}]}
{"messages": [{"role": "user", "content": "请问如果我想三日内游遍成都所有著名景点路线该如何规划？"}, {"role": "assistant", "content": "第一天，上午前往四川博物院，这是西南最大博物馆，可重点参观巴蜀青铜馆、汉代陶石馆等，欣赏说唱俑等必看展品，建议游览2-3小时。中午在附近品尝成都美食，如乐山鲜知味钵钵鸡等。下午参观杜甫草堂，游览正门、大廨、诗史堂等景点，感受杜甫故居的文化氛围，建议游览2小时左右。晚上前往宽窄巷子，在这里欣赏古街风貌，品尝成都小吃，还可以选择去蜀风雅韵观看川剧变脸等表演，感受成都的夜生活。 第二天上午前往武侯祠，感受三国文化，参观红墙竹影等著名景点，建议游览1-2小时。之后游览锦里古街，体验川西特色风情，品尝各种小吃，游览时间1-2小时。中午在锦里古街或附近就餐，品尝担担面、钟水饺等美食。下午从